In [7]:
import time
import csv
import re
import logging
from typing import Set, List, Tuple
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor
import warnings

import undetected_chromedriver as uc
from selenium.webdriver.chrome.options import Options

warnings.filterwarnings('ignore')

START_URL = 'https://www.livinginsider.com/searchword/all/all/2/%E0%B8%81%E0%B8%A3%E0%B8%B8%E0%B8%87%E0%B9%80%E0%B8%97%E0%B8%9E%E0%B8%A1%E0%B8%AB%E0%B8%B2%E0%B8%99%E0%B8%84%E0%B8%A3.html'
OUTPUT_CSV_FILE = 'livinginsider_listing_urls.csv'
MAX_PAGES = 50
PAGE_TIMEOUT = 30
SCROLL_ROUNDS = 15
SCROLL_DELAY = 0.4

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(), logging.FileHandler('scraper.log')]
)
logger = logging.getLogger(__name__)

@lru_cache(maxsize=128)
def parse_url_cached(url: str) -> Tuple:
    p = urlparse(url)
    return (p.scheme, p.netloc, p.path, p.params, p.query, p.fragment)

def build_page_url(url: str, page: int) -> str:
    scheme, netloc, path, params, query, fragment = parse_url_cached(url)
    path_segments = [seg for seg in path.split('/') if seg]
    
    page_idx = next((i for i, seg in enumerate(path_segments) 
                    if seg.isdigit()), None)
    
    if page_idx is not None:
        path_segments[page_idx] = str(page)
    elif page > 1:
        path_segments.append(str(page))
    
    new_path = '/' + '/'.join(path_segments)
    return urlunparse((scheme, netloc, new_path, params, query, fragment))

def get_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    opts = [
        "--no-sandbox",
        "--disable-dev-shm-usage",
        "--disable-gpu",
        "--disable-notifications",
        "--disable-web-resources",
        "--disable-images",
        "--disable-features=TranslateUI",
        "--blink-settings=imagesEnabled=false"
    ]
    
    for opt in opts:
        options.add_argument(opt)
    
    driver = uc.Chrome(options=options, suppress_welcome=True, version_main=None)
    driver.set_page_load_timeout(PAGE_TIMEOUT)
    return driver

def fast_collect_links(driver: uc.Chrome, base_url: str) -> List[str]:
    js_code = """
    const links = new Set();
    document.querySelectorAll("a[href*='/livingdetail/'][href$='.html']").forEach(el => {
        const href = el.getAttribute('href');
        if (href && !/(bclick|stories|banner)/.test(href)) {
            links.add(href.startsWith('/') ? arguments[0] + href : href);
        }
    });
    return Array.from(links);
    """
    try:
        return driver.execute_script(js_code, base_url) or []
    except Exception as e:
        logger.error(f"Link collection failed: {e}")
        return []

def smart_scroll(driver: uc.Chrome, max_rounds: int = SCROLL_ROUNDS) -> None:
    prev_height = driver.execute_script("return document.body.scrollHeight")
    
    for _ in range(max_rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
        time.sleep(SCROLL_DELAY)
        
        curr_height = driver.execute_script("return document.body.scrollHeight")
        if curr_height == prev_height:
            break
        prev_height = curr_height

def main() -> None:
    driver = None
    all_urls: Set[str] = set()
    last_signature = None
    base_domain = f"{urlparse(START_URL).scheme}://{urlparse(START_URL).netloc}"
    
    try:
        driver = get_driver()
        
        for page in range(1, MAX_PAGES + 1):
            target_url = build_page_url(START_URL, page)
            logger.info(f"Page {page}: {target_url}")
            
            try:
                driver.get(target_url)
                smart_scroll(driver)
                page_urls = fast_collect_links(driver, base_domain)
                
                if not page_urls:
                    logger.warning("Empty page detected")
                    break
                
                current_sig = page_urls[0]
                if last_signature == current_sig:
                    logger.info("Duplicate content detected - terminating")
                    break
                
                last_signature = current_sig
                prev_count = len(all_urls)
                all_urls.update(page_urls)
                
                logger.info(f"Page {page}: +{len(page_urls)} unique URLs | Total: {len(all_urls)}")
                
            except TimeoutError:
                logger.error(f"Timeout on page {page}")
                continue
            except Exception as e:
                logger.error(f"Page {page} error: {e}")
                continue
                
    except Exception as e:
        logger.critical(f"Driver initialization failed: {e}")
    finally:
        if driver:
            driver.quit()
    
    if all_urls:
        with open(OUTPUT_CSV_FILE, 'w', newline='', encoding='utf-8') as f:
            csv.writer(f).writerows([['ListingURL']] + [[url] for url in sorted(all_urls)])
        logger.info(f"Saved {len(all_urls)} URLs to {OUTPUT_CSV_FILE}")
    else:
        logger.error("No URLs collected")

if __name__ == "__main__":
    main()

[12:29:17] [INFO] patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
[12:29:18] [INFO] Page 1: https://www.livinginsider.com/searchword/all/all/1/%E0%B8%81%E0%B8%A3%E0%B8%B8%E0%B8%87%E0%B9%80%E0%B8%97%E0%B8%9E%E0%B8%A1%E0%B8%AB%E0%B8%B2%E0%B8%99%E0%B8%84%E0%B8%A3.html
[12:29:27] [WARNING] Empty page detected
[12:29:27] [ERROR] No URLs collected


In [7]:
# LivingInsider Scraper - Details

import csv
import time
import socket
from pathlib import Path
from urllib3.exceptions import ReadTimeoutError, ConnectionError as Urllib3ConnectionError
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, WebDriverException
from urllib.parse import urlparse

INPUT_CSV = "livinginsider_listing_urls.csv"
OUTPUT_CSV = "livinginsider_scraped_details.csv"
PAGE_TIMEOUT = 120
START_INDEX = 7192
RETRY_ATTEMPTS = 3
RETRY_DELAYS = [1, 2, 4]

def wait_ready(driver, timeout):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if driver.execute_script("return document.readyState") == "complete":
            return True
        time.sleep(0.2)
    return False

def click_expand(driver):
    try:
        expander = driver.find_elements(By.XPATH, "//a[@onclick='opendetile_living(this)']")
        if expander and expander[0].is_displayed():
            driver.execute_script("arguments[0].click();", expander[0])
            time.sleep(0.5)
    except:
        pass

def get_full_content(driver, url):
    try:
        click_expand(driver)
        
        title = ""
        price = ""
        size = ""
        description = ""
        location = ""
        
        title_elems = driver.find_elements(By.CSS_SELECTOR, "h1.font_sarabun.show-title")
        if title_elems:
            title = title_elems[0].text.strip()
        
        price_elems = driver.find_elements(By.CSS_SELECTOR, ".show_price_topic .price_topic b")
        if price_elems:
            price = price_elems[0].text.strip()
        
        size_elems = driver.find_elements(By.CSS_SELECTOR, ".detail_property_list_des_new .detail-property-list-text")
        if size_elems:
            size = size_elems[0].text.strip()
        
        desc_elems = driver.find_elements(By.CSS_SELECTOR, ".wordwrap, [id*='desc']")
        if desc_elems:
            description = desc_elems[0].text.strip()
        
        loc_elems = driver.find_elements(By.CSS_SELECTOR, ".form-group.group-location-detail .detail-text-zone a")
        if loc_elems:
            location = loc_elems[0].text.strip()
        
        full_content = f"""Title: {title}
Price: {price}
Size: {size}
Location: {location}

Description:
{description}"""
        
        return full_content.replace('\n', ' ').replace(',', ';')
        
    except Exception as e:
        print(f"  ⚠ Error extracting content: {str(e)}")
        return ""

def scrape_one(driver, url, csv_writer, csv_file):
    for attempt in range(RETRY_ATTEMPTS):
        try:
            print(f"  Scraping: {url[:60]}...", end="")
            driver.get(url)
            wait_ready(driver, PAGE_TIMEOUT)
            time.sleep(0.8)
            
            content = get_full_content(driver, url)
            
            if not content:
                print(f" ⚠ No content")
                return False
            
            csv_writer.writerow({
                "Post_URL": url,
                "Full_Content": content
            })
            csv_file.flush()
            print(f" ✓")
            return True
            
        except (ReadTimeoutError, socket.timeout):
            if attempt < RETRY_ATTEMPTS - 1:
                delay = RETRY_DELAYS[attempt]
                print(f" ⏱ Timeout, retry in {delay}s...", end="")
                time.sleep(delay)
            else:
                print(f" ✗ Timeout (after {RETRY_ATTEMPTS} attempts)")
                return False
                
        except Urllib3ConnectionError:
            if attempt < RETRY_ATTEMPTS - 1:
                delay = RETRY_DELAYS[attempt]
                print(f" 🔌 Connection error, retry in {delay}s...", end="")
                time.sleep(delay)
            else:
                print(f" ✗ Connection error (after {RETRY_ATTEMPTS} attempts)")
                return False
                
        except (TimeoutException, WebDriverException) as e:
            print(f" ✗ WebDriver error: {type(e).__name__}")
            return False
            
        except Exception as e:
            print(f" ✗ Unexpected error: {type(e).__name__}")
            return False
    
    return False

def main():
    if not Path(INPUT_CSV).exists():
        print(f"Input file {INPUT_CSV} not found")
        return
    
    with open(INPUT_CSV, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [r[0].strip() for r in reader if r and r[0].strip()]
    
    urls = urls[START_INDEX - 1:]
    
    print(f"Found {len(urls)} URLs to scrape (from index {START_INDEX})\n")
    
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-blink-features=AutomationControlled")
    
    driver = uc.Chrome(options=options, version_main=145)
    driver.set_page_load_timeout(60)
    
    file_exists = Path(OUTPUT_CSV).exists()
    
    with open(OUTPUT_CSV, "a" if file_exists else "w", newline="", encoding="utf-8") as f:
        if not file_exists:
            fieldnames = ["Post_URL", "Full_Content"]
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
        else:
            fieldnames = ["Post_URL", "Full_Content"]
            writer = csv.DictWriter(f, fieldnames=fieldnames)
        
        success_count = 0
        for i, u in enumerate(urls, 1):
            print(f"[{i}/{len(urls)} (#{START_INDEX + i - 1})]", end=" ")
            if scrape_one(driver, u, writer, f):
                success_count += 1
            time.sleep(0.3)
    
    driver.quit()
    
    print(f"\n✓ Done. {success_count} details saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

Found 6709 URLs to scrape (from index 7192)

[1/6709 (#7192)]   Scraping: https://www.livinginsider.com/livingdetail/2794855/2-story-t... ✓
[2/6709 (#7193)]   Scraping: https://www.livinginsider.com/livingdetail/2794882/%E0%B8%84... ✓
[3/6709 (#7194)]   Scraping: https://www.livinginsider.com/livingdetail/2794885/Townhome-... ✓
[4/6709 (#7195)]   Scraping: https://www.livinginsider.com/livingdetail/2795114/%E0%B8%82... ✓
[5/6709 (#7196)]   Scraping: https://www.livinginsider.com/livingdetail/2795130/%E0%B8%9A... ✓
[6/6709 (#7197)]   Scraping: https://www.livinginsider.com/livingdetail/2795160/Condomini... ✓
[7/6709 (#7198)]   Scraping: https://www.livinginsider.com/livingdetail/2795305/Condo-for... ✓
[8/6709 (#7199)]   Scraping: https://www.livinginsider.com/livingdetail/2795329/%E0%B8%8A... ✓
[9/6709 (#7200)]   Scraping: https://www.livinginsider.com/livingdetail/2795442/%E0%B8%9A... ✓
[10/6709 (#7201)]   Scraping: https://www.livinginsider.com/livingdetail/2795484/%E0%B8%82... ✓
[11/

In [ ]:
import json
import os
import re
import time
import csv
import logging
import random
import socket
import threading
import subprocess
import shutil
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, WebDriverException
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)

BASE_DIR = Path("/home/kongla/Documents/GitHub/Real-estate-Scraping")
OUTPUT_PATH = BASE_DIR / "livinginsider" / "livinginsider_final_output.csv"
PROFILE_PATH = BASE_DIR / "chrome_profile_li_deep"

START_URL = 'https://www.livinginsider.com/searchword/all/all/2/%E0%B8%81%E0%B8%A3%E0%B8%B8%E0%B8%87%E0%B9%80%E0%B8%97%E0%B8%9E%E0%B8%A1%E0%B8%AB%E0%B8%B2%E0%B8%99%E0%B8%84%E0%B8%A3.html'
MAX_PAGES = 5
PAGE_TIMEOUT = 60
AI_WORKERS = 10

MODEL_NAME = "typhoon-v2.5-30b-a3b-instruct"
client = OpenAI(api_key=os.getenv("TYPHOON_API_KEY"), base_url="https://api.opentyphoon.ai/v1")

OUTPUT_HEADERS = [
    "วันที่โพส", "website", "ประเภท", "สถานะ", "ชื่อโครงการ", 
    "ขนาด", "ราคา", "เขต", "Link", "เบอร์โทรศัพท์", "Line", "คำอธิบาย",
    "is_owner", "owner_confidence", "risk_flags"
]

SYSTEM_PROMPT = """คุณคือ AI วิเคราะห์อสังหาริมทรัพย์ สกัดข้อมูลเป็น JSON ตาม Schema ที่กำหนด
{
  "is_real_estate": true,
  "is_owner": true/false,
  "owner_confidence": 0.0,
  "post_date_text": "วันที่",
  "extracted": {
    "property_type": "", "rental_sale_status": "", "project_name": "", "district": "",
    "size_text": "", "price_text": "", "phone": "", "line": "", "description": ""
  }
}"""

csv_lock = threading.Lock()

def get_chrome_version():
    chrome_exec = shutil.which("google-chrome") or shutil.which("chromium-browser")
    try:
        res = subprocess.run([chrome_exec, "--version"], capture_output=True, text=True)
        return int(re.search(r"(\d+)\.", res.stdout).group(1))
    except: return 0

def create_driver():
    PROFILE_PATH.mkdir(parents=True, exist_ok=True)
    opts = uc.ChromeOptions()
    opts.add_argument(f"--user-data-dir={PROFILE_PATH}")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--disable-notifications")
    opts.page_load_strategy = "eager"
    return uc.Chrome(options=opts, version_main=get_chrome_version())

def build_page_url(u, page):
    p = urlparse(u)
    parts = [x for x in p.path.split('/') if x]
    k = next((i for i, seg in enumerate(parts) if re.fullmatch(r'\d+', seg)), None)
    if k is None:
        np = p.path.rstrip('/') if page <= 1 else p.path.rstrip('/') + f'/{page}'
    else:
        parts[k] = '1' if page <= 1 else str(page)
        np = '/' + '/'.join(parts)
    return urlunparse((p.scheme, p.netloc, np, p.params, urlencode(dict(parse_qsl(p.query)), doseq=True), p.fragment))

def extract_listing_content_js(driver):
    return driver.execute_script("""
        const expand = document.querySelector("a[onclick*='opendetile_living']");
        if(expand) expand.click();
        
        const getTxt = (sel) => document.querySelector(sel)?.innerText?.trim() || "";
        
        return {
            title: getTxt("h1.font_sarabun.show-title"),
            price: getTxt(".show_price_topic .price_topic b"),
            size: getTxt(".detail_property_list_des_new .detail-property-list-text"),
            location: getTxt(".form-group.group-location-detail .detail-text-zone a"),
            description: getTxt(".wordwrap, [id*='desc']")
        };
    """)

def call_llm_service(payload: str) -> dict | None:
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=0.0,
                max_tokens=15000,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": payload},
                ],
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except Exception:
            time.sleep(2 ** attempt)
    return None

def ai_processor(raw_data, url):
    payload = f"URL: {url}\nData: {json.dumps(raw_data, ensure_ascii=False)}"
    data = call_llm_service(payload)
    if data and data.get("is_real_estate"):
        ext = data.get("extracted", {})
        row = {
            "วันที่โพส": data.get("post_date_text", "-"),
            "website": "livinginsider",
            "ประเภท": ext.get("property_type", "-"),
            "สถานะ": ext.get("rental_sale_status", "-"),
            "ชื่อโครงการ": ext.get("project_name", "-"),
            "ขนาด": ext.get("size_text", "-"),
            "ราคา": ext.get("price_text", "-"),
            "เขต": ext.get("district", "-"),
            "Link": url,
            "เบอร์โทรศัพท์": ext.get("phone", "-"),
            "Line": ext.get("line", "-"),
            "คำอธิบาย": ext.get("description", "-"),
            "is_owner": data.get("is_owner", False),
            "owner_confidence": data.get("owner_confidence", 0),
            "risk_flags": str(data.get("risk_flags", []))
        }
        with csv_lock:
            with open(OUTPUT_PATH, 'a', encoding='utf-8-sig', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=OUTPUT_HEADERS)
                writer.writerow(row)

def main():
    if not OUTPUT_PATH.exists():
        OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
        with open(OUTPUT_PATH, 'w', encoding='utf-8-sig', newline='') as f:
            csv.DictWriter(f, fieldnames=OUTPUT_HEADERS).writeheader()

    seen_urls = set()
    with open(OUTPUT_PATH, 'r', encoding='utf-8-sig') as f:
        seen_urls.update(r['Link'] for r in csv.DictReader(f) if r.get('Link'))

    driver = create_driver()
    try:
        with ThreadPoolExecutor(max_workers=AI_WORKERS) as executor:
            for page in range(1, MAX_PAGES + 1):
                logger.info(f"Directory Page {page}")
                driver.get(build_page_url(START_URL, page))
                time.sleep(5)
                
                urls = driver.execute_script("""
                    return Array.from(document.querySelectorAll("a[href*='/detail/'], a[href*='/livingdetail/']"))
                        .map(a => a.href.split('?')[0])
                        .filter((v, i, a) => a.indexOf(v) === i && !v.includes('stories'));
                """)
                
                for url in urls:
                    if url in seen_urls: continue
                    
                    try:
                        logger.info(f"Deep Scraping: {url}")
                        driver.get(url)
                        time.sleep(random.uniform(1.5, 3.0))
                        content = extract_listing_content_js(driver)
                        if content['title']:
                            executor.submit(ai_processor, content, url)
                            seen_urls.add(url)
                    except Exception as e:
                        logger.error(f"Listing Error: {e}")
                        continue
    finally:
        if driver:
            driver.quit()

if __name__ == "__main__":
    main()

[12:41:21] [INFO] patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
[12:41:23] [INFO] Directory Page 1
[12:41:33] [INFO] Deep Scraping: https://www.livinginsider.com/detail/land-for-sale-3031806
[12:41:38] [INFO] Deep Scraping: https://www.livinginsider.com/detail/house-for-sale-burasiri-rama-2-4bedroom-2947796
[12:41:42] [INFO] Deep Scraping: https://www.livinginsider.com/detail/condo-for-sale-1bedroom-2887617
[12:41:47] [INFO] Deep Scraping: https://www.livinginsider.com/detail/condo-for-sale-flexi-sathon-charoen-nakhon-1bedroom-2979278
[12:41:51] [INFO] Deep Scraping: https://www.livinginsider.com/detail/house-for-sale-baan-lalin-in-the-park-watcharapol-paholyothin-3bedroom-2941223
[12:41:56] [INFO] HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
[12:41:56] [INFO] Deep Scraping: https://www.livinginsider.com/detail/homeoffice-for-sale-2113700
[12:41:58] [INFO] HTTP Request: POST https://api.openty